# Assignment 4: Embeddings → ML model → Transformer → Fine-tuning (Adilet Search)

**Task.** Query–article relevance classification, used as the re-ranking stage of the legal search system. For a question and a candidate article of Kazakhstan legislation (Russian or Kazakh), predict whether the article answers the question.

- **X:** a pair (question, candidate article). **y:** 1 if the question was written for that article, else 0.
- **Candidates:** the top 20 articles of a tuned BM25, the same for every configuration, so classification and ranking metrics are measured on identical pairs.

| ID | Representation | Model |
|---|---|---|
| **E0** baseline | TF-IDF and lexical-overlap features of the pair | Logistic Regression |
| **E1** embeddings → ML | dense sentence embeddings (`multilingual-e5-base`, 768 dimensions): `|u−v|`, `u⊙v`, cosine | Logistic Regression; gradient boosting |
| **E2** transformer, no fine-tuning | the cross-encoder's own subword tokens, question and article read together | pretrained multilingual cross-encoder, zero-shot |
| **E3** fine-tuned transformer | same | the same cross-encoder, fine-tuned on our pairs |

**Research questions**

- **RQ1.** Do dense embeddings beat the baseline representation (E1 vs E0)?
- **RQ2.** How much better is a pretrained transformer than classical ML on embeddings (E2 vs E1)?
- **RQ3.** How much does fine-tuning help on this domain, statistically and practically (E3 vs E2)?
- **RQ4.** What does each step cost (parameters, training time, inference time), and which configuration should the product use?

**Run on a GPU runtime** (Colab: Runtime → Change runtime type → T4 GPU), then *Run all*. No tokens or logins are needed.

In [ ]:
# §0 Setup. The project repository is public; the corpus is downloaded from Zenodo and verified.
import os, subprocess, sys

# Colab also ships TensorFlow; if transformers touches it, TensorFlow reserves the whole GPU.
# bm25s runs a JAX operation when it is imported, and JAX then reserves 75% of the GPU for itself.
# Keep JAX on the CPU: the GPU is for PyTorch only.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["USE_TF"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/R0onen/adilet-search.git"
REPO_REF = os.environ.get("ADILET_REF", "main")
if os.path.exists("ml/scripts/download_data.py"):  # already inside the repository
    REPO_DIR = os.getcwd()
elif os.path.exists("../scripts/download_data.py"):  # opened from ml/notebooks
    REPO_DIR = os.path.abspath("../..")
else:
    REPO_DIR = os.path.abspath("adilet-search")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR], check=True)

# Colab already has torch, transformers, scikit-learn and pandas; these are the extras.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bm25s==0.3.13", "PyStemmer==3.1.0", "peft", "pyarrow"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", os.path.join(REPO_DIR, "ml")], check=True)
# Colab preinstalls an old torchao that peft refuses to work next to; nothing here needs it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
sys.path.insert(0, os.path.join(REPO_DIR, "ml", "src"))
print("repository:", REPO_DIR)

In [ ]:
import gc, json, math, random, time
from pathlib import Path

import bm25s
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import peft
import sklearn
import torch
import transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from adilet_ml.retrieval.sparse import tokenize  # the product's tokenizer (Russian stemming)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# SMOKE=1 shrinks everything for a CPU check of the code; the reported numbers need SMOKE=0.
SMOKE = os.environ.get("SMOKE", "0") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REPO = Path(REPO_DIR)
OUT = Path("a4_results"); OUT.mkdir(exist_ok=True)

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | scikit-learn {sklearn.__version__} | pandas {pd.__version__} | bm25s {bm25s.__version__}")
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU (training will be slow)")
print("seed:", SEED, "| smoke run:", SMOKE)
if DEVICE == "cuda":
    free_gb, total_gb = (x / 2**30 for x in torch.cuda.mem_get_info())
    print(f"GPU memory: {free_gb:.1f} GiB free of {total_gb:.1f} GiB")
    # Something else in this session holds the GPU (an earlier run, another library): training would run out of memory.
    assert free_gb > 0.8 * total_gb, "The GPU is not free. Use Runtime -> Restart session, then Run all."

## §1 Data, split and leakage checks

- **Corpus:** AdiletCodex v1.0 (CC BY 4.0, doi:10.5281/zenodo.22812626): the 8 main codes in Russian and Kazakh, 9,259 articles.
- **Questions:** LLM-written questions (`plain`: everyday wording, `term`: legal terminology) for sampled articles; article titles are added to the **training** set only.
- **Split:** by article, fixed in `data/eval/splits.json` (seed 42). Validation and test contain synthetic questions only.
- **Leakage checks** (asserted in the code below): no article is in two splits; the TF-IDF vocabulary is fitted on the corpus, not on questions; thresholds and hyperparameters are chosen on train/validation only; hard negatives for training come from training questions only.

In [ ]:
if not (REPO / "data/processed/chunks.parquet").exists():
    subprocess.run([sys.executable, str(REPO / "ml/scripts/download_data.py")], check=True)

chunks = pd.read_parquet(REPO / "data/processed/chunks.parquet",
                         columns=["article_id", "lang", "chunk_index", "text", "unit_title", "doc_short_title"])
articles = (chunks.sort_values(["article_id", "chunk_index"])
            .groupby("article_id", sort=False)
            .agg(lang=("lang", "first"), body=("text", "\n".join), title=("unit_title", "first"), doc=("doc_short_title", "first"))
            .reset_index())
articles["title"] = articles["title"].fillna("")
# What the models read: the code's name, then the article body. The title is left out on purpose:
# title queries would otherwise be answered by string matching.
articles["passage"] = articles["doc"] + ". " + articles["body"].str.slice(0, 2000)
corpus = {lang: g.reset_index(drop=True) for lang, g in articles.groupby("lang")}
print({lang: len(g) for lang, g in corpus.items()}, "articles")

def read_jsonl(path):
    return pd.DataFrame([json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line])

synthetic = read_jsonl(REPO / "data/eval/synthetic/queries.jsonl").merge(read_jsonl(REPO / "data/eval/synthetic/qrels.jsonl"), on="query_id")
titles = read_jsonl(REPO / "data/eval/queries.jsonl").merge(read_jsonl(REPO / "data/eval/qrels.jsonl"), on="query_id")
title_split = {qid: split for split, ids in json.loads((REPO / "data/eval/splits.json").read_text(encoding="utf-8"))["query_splits"].items() for qid in ids}
titles["split"] = titles["query_id"].map(title_split)
titles["query_type"] = "title"

N_TITLE_TRAIN = 200 if SMOKE else 2500
title_train = titles[titles["split"] == "train"].sample(N_TITLE_TRAIN, random_state=SEED)
queries = pd.concat([synthetic, title_train], ignore_index=True)[["query_id", "text", "lang", "query_type", "split", "article_id"]]
if SMOKE:
    queries = queries.groupby(["split", "lang"]).head(8).reset_index(drop=True)

print(queries.groupby(["split", "query_type", "lang"]).size().unstack(fill_value=0))
display(queries[queries["query_type"] != "title"].groupby("query_type").head(2)[["text", "lang", "query_type"]])

# Leakage check: the article behind a question decides its split, so no article may appear in two splits.
unit = queries["article_id"].str.replace(r":(ru|kk):", ":", regex=True)
assert queries.groupby(unit)["split"].nunique().max() == 1, "an article is in two splits"
print("leakage check OK: every article (both languages) is in exactly one split")

## §2 Candidates and metrics

BM25 (parameters tuned on validation) retrieves the top 20 articles per question. Training keeps the positive and 7 hard negatives per question; validation and test keep all 20 candidates. Metrics: **ROC-AUC, PR-AUC, F1** (threshold chosen on validation) for the classifier; **nDCG@10, MRR@10** for the ranking.

In [ ]:
K_CAND, N_NEG = 20, 7

def bm25_tokens(texts, lang):
    return [tokenize(t, lang) for t in texts]

corpus_tokens = {lang: bm25_tokens(c["body"], lang) for lang, c in corpus.items()}
article_pos = {lang: {a: i for i, a in enumerate(c["article_id"])} for lang, c in corpus.items()}

def bm25_index(lang, k1=1.2, b=0.75):
    index = bm25s.BM25(k1=k1, b=b)
    index.index(corpus_tokens[lang], show_progress=False)
    return index

def retrieve(index, lang, texts, k=K_CAND):
    docs, scores = index.retrieve(bm25_tokens(texts, lang), k=k, show_progress=False, backend_selection="numpy")
    return docs, scores

def rank_metrics(ranks):
    # rank of the correct article (1-based; inf = not among the candidates)
    ranks = np.asarray(ranks, dtype=float)
    hit = ranks <= 10
    return {"nDCG@10": float(np.where(hit, 1 / np.log2(ranks + 1), 0).mean()),
            "MRR@10": float(np.where(hit, 1 / ranks, 0).mean()),
            "Recall@10": float(hit.mean())}

assert abs(rank_metrics([1, 2, np.inf])["nDCG@10"] - (1 + 1 / math.log2(3)) / 3) < 1e-9
print("metric check OK")

In [ ]:
def bm25_ranks(index, lang, frame):
    docs, _ = retrieve(index, lang, frame["text"].tolist())
    target = frame["article_id"].map(article_pos[lang]).to_numpy()
    hits = docs == target[:, None]
    return np.where(hits.any(1), hits.argmax(1) + 1, np.inf)

val_q = queries[queries["split"] == "val"]
grid_rows, best_bm25 = [], {}
for lang in ("ru", "kk"):
    frame = val_q[val_q["lang"] == lang]
    for k1 in [0.6, 0.9, 1.2, 1.5, 2.0]:
        for b in [0.3, 0.5, 0.75, 0.9]:
            grid_rows.append({"lang": lang, "k1": k1, "b": b, **rank_metrics(bm25_ranks(bm25_index(lang, k1, b), lang, frame))})
grid = pd.DataFrame(grid_rows)
for lang in ("ru", "kk"):
    best = grid[grid["lang"] == lang].sort_values("nDCG@10", ascending=False).iloc[0]
    default = grid[(grid["lang"] == lang) & (grid["k1"] == 1.2) & (grid["b"] == 0.75)].iloc[0]
    best_bm25[lang] = (float(best["k1"]), float(best["b"]))
    print(f"{lang}: best k1={best['k1']}, b={best['b']} (val nDCG@10 {best['nDCG@10']:.4f}); default 1.2 / 0.75: {default['nDCG@10']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, lang in zip(axes, ("ru", "kk")):
    table = grid[grid["lang"] == lang].pivot(index="k1", columns="b", values="nDCG@10")
    image = ax.imshow(table.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(table.columns)), table.columns); ax.set_yticks(range(len(table.index)), table.index)
    ax.set_xlabel("b (length normalisation)"); ax.set_ylabel("k1 (term saturation)")
    ax.set_title(f"BM25 validation nDCG@10, {'Russian' if lang == 'ru' else 'Kazakh'}")
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            ax.text(j, i, f"{table.values[i, j]:.3f}", ha="center", va="center", color="white", fontsize=8)
    fig.colorbar(image, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# Candidate pairs from the tuned BM25: this table is the input of every model below.
def build_pairs(frame, n_neg=None):
    rows = []
    for lang in ("ru", "kk"):
        part = frame[frame["lang"] == lang]
        if part.empty:
            continue
        docs, scores = retrieve(bm25_index(lang, *best_bm25[lang]), lang, part["text"].tolist())
        ids = corpus[lang]["article_id"].to_numpy()
        for (_, q), cand, score in zip(part.iterrows(), docs, scores):
            kept_neg = 0
            for rank, (d, s) in enumerate(zip(cand, score), 1):
                label = int(ids[d] == q["article_id"])
                if n_neg is not None and not label:
                    if kept_neg >= n_neg:
                        continue
                    kept_neg += 1
                rows.append((q["query_id"], q["text"], lang, q["query_type"], int(d), float(s), rank, label))
            if n_neg is not None and q["article_id"] not in ids[cand]:  # training always sees the positive
                rows.append((q["query_id"], q["text"], lang, q["query_type"], article_pos[lang][q["article_id"]], 0.0, K_CAND + 1, 1))
    pairs = pd.DataFrame(rows, columns=["query_id", "query", "lang", "query_type", "doc", "bm25", "bm25_rank", "label"])
    pairs["passage"] = [corpus[l]["passage"].iat[d] for l, d in zip(pairs["lang"], pairs["doc"])]
    return pairs

train_pairs = build_pairs(queries[queries["split"] == "train"], n_neg=N_NEG)
val_pairs = build_pairs(queries[queries["split"] == "val"])
test_pairs = build_pairs(queries[queries["split"] == "test"])
for name, p in (("train", train_pairs), ("val", val_pairs), ("test", test_pairs)):
    print(f"{name}: {len(p):,} pairs, {p['query_id'].nunique():,} questions, {p['label'].mean():.1%} positive")
ceiling = test_pairs.groupby("query_id")["label"].max().mean()
print(f"Ceiling: the correct article is among the {K_CAND} BM25 candidates for {ceiling:.1%} of the test questions")
print(f"Class balance: 1 positive per {(1 - test_pairs['label'].mean()) / test_pairs['label'].mean():.0f} negatives on test "
      f"-> class weights in the models, threshold tuned on validation.")

In [ ]:
# One evaluation function for every model: scores for the test pairs -> classifier + ranking metrics.
RESULTS, SCORES = [], {}

def evaluate(name, family, scores, pairs=None, threshold=None, extra=None):
    pairs = test_pairs if pairs is None else pairs
    frame = pairs.assign(score=np.asarray(scores, dtype=float))
    order = frame.sort_values(["query_id", "score"], ascending=[True, False])
    order["rank"] = order.groupby("query_id").cumcount() + 1
    ranks = order[order["label"] == 1].set_index("query_id")["rank"].reindex(frame["query_id"].unique()).fillna(np.inf)
    row = {"model": name, "family": family, **rank_metrics(ranks.to_numpy())}
    if threshold is not None:
        pred = (frame["score"] >= threshold).astype(int)
        row.update({"F1": f1_score(frame["label"], pred), "Precision": precision_score(frame["label"], pred, zero_division=0),
                    "Recall": recall_score(frame["label"], pred), "ROC-AUC": roc_auc_score(frame["label"], frame["score"])})
    row.update(extra or {})
    if pairs is test_pairs:
        RESULTS[:] = [r for r in RESULTS if r["model"] != name] + [row]
        SCORES[name] = frame["score"].to_numpy()
    return row

def best_threshold(scores, labels):
    # the decision threshold is chosen on validation pairs, never on test
    grid = np.quantile(scores, np.linspace(0.5, 0.995, 60))
    return float(max(grid, key=lambda t: f1_score(labels, scores >= t)))

# BM25 itself as a ranker: default parameters, then tuned
default_rows = []
for lang in ("ru", "kk"):
    part = queries[(queries["split"] == "test") & (queries["lang"] == lang)]
    default_rows.append(bm25_ranks(bm25_index(lang), lang, part))
RESULTS.append({"model": "BM25 (default k1=1.2, b=0.75)", "family": "classical", **rank_metrics(np.concatenate(default_rows))})
print(evaluate("BM25 (tuned)", "classical", test_pairs["bm25"]))

## §3 E0: baseline representation (TF-IDF features + Logistic Regression)

Eight numbers describe a pair: BM25 score and reciprocal rank, word and character TF-IDF similarity, query-term coverage, Jaccard overlap, lengths. No meaning beyond shared strings.

In [ ]:
vectorizers = {}
for lang, c in corpus.items():
    word = TfidfVectorizer(tokenizer=lambda t, lang=lang: tokenize(t, lang), token_pattern=None, lowercase=False,
                           sublinear_tf=True, min_df=2, dtype=np.float32)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=3, max_features=200_000, dtype=np.float32)
    vectorizers[lang] = (word, word.fit_transform(c["body"]), char, char.fit_transform(c["body"].str.slice(0, 3000)))
token_sets = {lang: [set(t) for t in toks] for lang, toks in corpus_tokens.items()}

FEATURES = ["bm25", "bm25_rr", "tfidf_word", "tfidf_char", "coverage", "jaccard", "len_doc", "len_query"]

def add_features(pairs):
    out = pairs.copy()
    for column in FEATURES[2:]:
        out[column] = 0.0
    out["bm25_rr"] = 1.0 / out["bm25_rank"]
    for lang in ("ru", "kk"):
        mask = (out["lang"] == lang).to_numpy()
        if not mask.any():
            continue
        word, word_matrix, char, char_matrix = vectorizers[lang]
        part = out[mask]
        unique = part["query"].drop_duplicates().tolist()
        position = {q: i for i, q in enumerate(unique)}
        q_index = part["query"].map(position).to_numpy()
        docs = part["doc"].to_numpy()
        q_word, q_char = word.transform(unique), char.transform(unique)
        out.loc[mask, "tfidf_word"] = np.asarray(q_word[q_index].multiply(word_matrix[docs]).sum(1)).ravel()
        out.loc[mask, "tfidf_char"] = np.asarray(q_char[q_index].multiply(char_matrix[docs]).sum(1)).ravel()
        q_sets = [set(tokenize(q, lang)) for q in unique]
        inter = np.array([len(q_sets[i] & token_sets[lang][d]) for i, d in zip(q_index, docs)])
        q_len = np.array([max(len(q_sets[i]), 1) for i in q_index])
        union = np.array([max(len(q_sets[i] | token_sets[lang][d]), 1) for i, d in zip(q_index, docs)])
        out.loc[mask, "coverage"] = inter / q_len
        out.loc[mask, "jaccard"] = inter / union
        out.loc[mask, "len_doc"] = np.log1p([len(corpus_tokens[lang][d]) for d in docs])
        out.loc[mask, "len_query"] = np.log1p(q_len)
    return out

train_f, val_f, test_f = add_features(train_pairs), add_features(val_pairs), add_features(test_pairs)
display(train_f.groupby("label")[FEATURES].mean().round(3))

In [ ]:
y_train, groups = train_f["label"], train_f["query_id"]
proba = lambda m, X: m.predict_proba(X)[:, 1]
COST = {}

def fit_eval(name, family, model, X_tr, X_val, X_te, extra=None, fit_params=None):
    started = time.time(); model.fit(X_tr, y_train, **(fit_params or {})); minutes = (time.time() - started) / 60
    val_scores = proba(model, X_val)
    threshold = best_threshold(val_scores, val_pairs["label"].to_numpy())
    started = time.time(); test_scores = proba(model, X_te)
    COST[name] = {"train minutes": minutes, "inference ms / 20 candidates": (time.time() - started) / len(X_te) * 20 * 1000}
    return evaluate(name, family, test_scores, threshold=threshold, extra={"train minutes": minutes, **(extra or {})})

e0 = Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, random_state=SEED))])
print(fit_eval("E0 TF-IDF features + LogReg", "baseline", e0, train_f[FEATURES], val_f[FEATURES], test_f[FEATURES],
               extra={"trainable params": len(FEATURES) + 1}))

## §4 E1: dense embeddings → classical ML

**Embedding model:** `intfloat/multilingual-e5-base`, the encoder the product uses for retrieval. 768 dimensions, mean pooling, L2-normalised; questions get the prefix `query: `, articles `passage: ` (as the model was trained). It is **contextual** (a word's vector depends on its sentence) and multilingual, so a plain-language question can be close to an article that shares no words with it.

**Pair features:** `|u − v|`, `u ⊙ v` (1,536 numbers) and the cosine similarity. Models on top:

- the cosine alone, with no training (what the embedding space already knows);
- Logistic Regression on the embedding features only (pure "embeddings → ML");
- gradient boosting on the cosine plus the E0 lexical features (does the dense signal add to the lexical one?), tuned with a randomised search under GroupKFold.

In [ ]:
from transformers import AutoModel, AutoTokenizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

EMB_NAME = "intfloat/multilingual-e5-base"
EMB_MAX_LEN = 128 if SMOKE else 512
emb_tokenizer = AutoTokenizer.from_pretrained(EMB_NAME)
emb_model = AutoModel.from_pretrained(EMB_NAME).to(DEVICE).eval()

@torch.inference_mode()
def embed(texts, prefix, batch_size=64):
    order = np.argsort([len(t) for t in texts])  # similar lengths together: less padding
    out = np.zeros((len(texts), emb_model.config.hidden_size), dtype=np.float32)
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]
        batch = emb_tokenizer([prefix + texts[i] for i in idx], truncation=True, max_length=EMB_MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE == "cuda"):
            hidden = emb_model(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1)
        pooled = (hidden * mask).sum(1) / mask.sum(1)
        out[idx] = torch.nn.functional.normalize(pooled.float(), dim=-1).cpu().numpy()
    return out

all_pairs = pd.concat([train_pairs, val_pairs, test_pairs], ignore_index=True)
unique_q = all_pairs["query"].drop_duplicates().tolist()
unique_p = all_pairs[["lang", "doc", "passage"]].drop_duplicates(["lang", "doc"])
started = time.time()
Q = embed(unique_q, "query: ")
P = embed(unique_p["passage"].tolist(), "passage: ")
embed_minutes = (time.time() - started) / 60
q_pos = {q: i for i, q in enumerate(unique_q)}
p_pos = {key: i for i, key in enumerate(zip(unique_p["lang"], unique_p["doc"]))}
print(f"Embedded {len(unique_q):,} questions and {len(unique_p):,} articles: dimension {Q.shape[1]}, {embed_minutes:.1f} min on {DEVICE}")

def emb_features(pairs):
    u = Q[pairs["query"].map(q_pos).to_numpy()]
    v = P[[p_pos[key] for key in zip(pairs["lang"], pairs["doc"])]]
    cos = (u * v).sum(1, keepdims=True)
    return np.hstack([np.abs(u - v), u * v, cos]), cos.ravel()

E_train, cos_train = emb_features(train_pairs)
E_val, cos_val = emb_features(val_pairs)
E_test, cos_test = emb_features(test_pairs)
embed_params = sum(p.numel() for p in emb_model.parameters())
del emb_model; gc.collect(); torch.cuda.empty_cache()

print(evaluate("E1a e5 cosine similarity (no training)", "embeddings", cos_test,
               threshold=best_threshold(cos_val, val_pairs["label"].to_numpy()), extra={"trainable params": 0}))

cv = GroupKFold(n_splits=3)
lr_emb = GridSearchCV(Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=300 if SMOKE else 1000, class_weight="balanced", random_state=SEED))]),
                      {"clf__C": [0.01] if SMOKE else [0.001, 0.01, 0.1]}, scoring="roc_auc", cv=cv, n_jobs=1)
row = fit_eval("E1b e5 embeddings + LogReg", "embeddings", lr_emb, E_train, E_val, E_test,
               extra={"trainable params": E_train.shape[1] + 1}, fit_params={"groups": groups})
print("LogReg on embeddings: best", lr_emb.best_params_, "| CV ROC-AUC", round(lr_emb.best_score_, 4)); print(row)

G_cols = ["e5_cos"] + FEATURES
G_train, G_val, G_test = (f.assign(e5_cos=c)[G_cols] for f, c in ((train_f, cos_train), (val_f, cos_val), (test_f, cos_test)))
gb = RandomizedSearchCV(HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
                        {"learning_rate": [0.03, 0.05, 0.1, 0.2], "max_iter": [100, 200, 400], "max_leaf_nodes": [7, 15, 31],
                         "min_samples_leaf": [10, 20, 50], "l2_regularization": [0.0, 0.1, 1.0]},
                        n_iter=2 if SMOKE else 20, scoring="roc_auc", cv=cv, random_state=SEED, n_jobs=1)
row = fit_eval("E1c e5 cosine + lexical features + gradient boosting", "embeddings", gb, G_train, G_val, G_test, fit_params={"groups": groups})
print("Gradient boosting: best", gb.best_params_, "| CV ROC-AUC", round(gb.best_score_, 4)); print(row)

## §5 E2: pretrained transformer without fine-tuning

**Model:** `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`: 12 layers, 118 M parameters, trained on mMARCO (passage ranking in 14 languages, Russian included, Kazakh not). **Why this one:** it is multilingual; as a *cross-encoder* it reads the question and the article in one sequence, so every question token attends to every article token (E1 compares two vectors computed separately); and it is small enough to rerank on a CPU in the product. It uses its own SentencePiece subword tokenizer; the article is truncated, never the question.

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

CE_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
MAX_LEN = 128 if SMOKE else 320
FP16 = DEVICE == "cuda"

class PairDataset(Dataset):
    # (question, article) pairs tokenised together, truncating only the article
    def __init__(self, pairs, tokenizer, max_len=MAX_LEN, float_labels=True):
        self.encoded = tokenizer(pairs["query"].tolist(), pairs["passage"].tolist(), truncation="only_second", max_length=max_len)
        self.labels = pairs["label"].astype("float32" if float_labels else "int64").to_numpy()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return {**{k: v[i] for k, v in self.encoded.items()}, "labels": self.labels[i]}

class BCETrainer(Trainer):
    # One output logit, binary cross-entropy loss (the standard cross-encoder objective)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(outputs.logits.view(-1), labels.float().view(-1))
        return (loss, outputs) if return_outputs else loss

def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def predict_scores(trainer, dataset, binary_logit=True):
    logits = trainer.predict(dataset).predictions
    logits = np.asarray(logits[0] if isinstance(logits, tuple) else logits, dtype=np.float32)
    if not np.isfinite(logits).all():  # half precision can overflow; say so instead of failing in the metrics
        print(f"WARNING: {int((~np.isfinite(logits)).sum())} non-finite scores were clipped")
        logits = np.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)
    return logits.reshape(-1) if binary_logit else logits[:, 1] - logits[:, 0]

def training_args(name, lr, epochs, batch=32, **kw):
    return TrainingArguments(output_dir=f"runs/{name}", learning_rate=lr, num_train_epochs=epochs,
                             per_device_train_batch_size=batch, per_device_eval_batch_size=64,
                             warmup_steps=int(0.1 * math.ceil(len(train_pairs) / batch) * epochs), weight_decay=0.01, fp16=FP16, seed=SEED, logging_steps=50,
                             save_strategy="no", report_to="none", disable_tqdm=True, **kw)

ce_tokenizer = AutoTokenizer.from_pretrained(CE_NAME)
ce_train = PairDataset(train_pairs, ce_tokenizer)
ce_val, ce_test = PairDataset(val_pairs, ce_tokenizer), PairDataset(test_pairs, ce_tokenizer)
collator = DataCollatorWithPadding(ce_tokenizer)

def evaluate_transformer(name, family, trainer, val_set, test_set, binary_logit=True, extra=None):
    val_scores = predict_scores(trainer, val_set, binary_logit)
    threshold = best_threshold(val_scores, val_pairs["label"].to_numpy())
    val_row = evaluate(name, family, val_scores, pairs=val_pairs)
    row = evaluate(name, family, predict_scores(trainer, test_set, binary_logit), threshold=threshold,
                   extra={"val nDCG@10": val_row["nDCG@10"], **(extra or {})})
    return row


def timed_eval(name, family, trainer, extra=None):
    started = time.time()
    row = evaluate_transformer(name, family, trainer, ce_val, ce_test, extra=extra)
    COST.setdefault(name, {})["inference ms / 20 candidates"] = (time.time() - started) / (len(ce_val) + len(ce_test)) * 20 * 1000
    return row

zero_model = AutoModelForSequenceClassification.from_pretrained(CE_NAME).to(DEVICE)
zero_trainer = BCETrainer(model=zero_model, args=training_args("zero", 0.0, 0), data_collator=collator)
ce_params = count_parameters(zero_model)[1]
print(timed_eval("E2 cross-encoder, zero-shot", "transformer", zero_trainer, extra={"trainable params": 0}))
COST["E2 cross-encoder, zero-shot"]["train minutes"] = 0.0
del zero_model, zero_trainer; gc.collect(); torch.cuda.empty_cache()

## §6 E3: the same transformer, fine-tuned on our pairs

| Setting | Value |
|---|---|
| Loss | binary cross-entropy on one output logit |
| Optimizer | AdamW, weight decay 0.01, linear warm-up over the first 10% of steps |
| Learning rate | 2e-5 (Assignment 3 searches 1e-5 / 2e-5 / 5e-5; another value can be set with the `FT_LR` environment variable) |
| Batch size, epochs | 32, 2 |
| Max length | 320 tokens |
| Model selection | validation nDCG@10 after each epoch; the best epoch is kept |

In [ ]:
FT_LR = float(os.environ.get("FT_LR", "2e-5"))
FT = {"learning_rate": FT_LR, "epochs": 1 if SMOKE else 2, "batch_size": 32, "max_len": MAX_LEN,
      "optimizer": "AdamW (weight_decay 0.01, warmup 10%)", "loss": "binary cross-entropy"}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
ft_model = AutoModelForSequenceClassification.from_pretrained(CE_NAME).to(DEVICE)
history, best_val = [], -1.0
started = time.time()
for epoch in range(1, FT["epochs"] + 1):
    # one Trainer per epoch keeps the code simple; the learning-rate schedule restarts each epoch
    trainer = BCETrainer(model=ft_model, args=training_args(f"e3-epoch{epoch}", FT_LR, 1), train_dataset=ce_train, data_collator=collator)
    trainer.train()
    val_ndcg = evaluate("val", "ft", predict_scores(trainer, ce_val), pairs=val_pairs)["nDCG@10"]
    history.append({"epoch": epoch, "val nDCG@10": val_ndcg}); print(history[-1])
    if val_ndcg > best_val:
        best_val = val_ndcg
        trainer.save_model(str(OUT / "reranker-ft")); ce_tokenizer.save_pretrained(str(OUT / "reranker-ft"))
    del trainer
ft_minutes = (time.time() - started) / 60
del ft_model; gc.collect(); torch.cuda.empty_cache()

ft_model = AutoModelForSequenceClassification.from_pretrained(str(OUT / "reranker-ft")).to(DEVICE)
ft_trainer = BCETrainer(model=ft_model, args=training_args("e3-eval", 0.0, 0), data_collator=collator)
print(timed_eval("E3 cross-encoder, fine-tuned", "transformer", ft_trainer, extra={"trainable params": ce_params, "train minutes": ft_minutes}))
COST["E3 cross-encoder, fine-tuned"]["train minutes"] = ft_minutes
del ft_model, ft_trainer; gc.collect(); torch.cuda.empty_cache()

## §7 Results: one table, the same test pairs for every configuration

In [ ]:
results = pd.DataFrame(RESULTS)
labels = test_pairs["label"].to_numpy()
results["PR-AUC"] = [average_precision_score(labels, SCORES[m]) if m in SCORES else np.nan for m in results["model"]]
results["model size (params)"] = [ce_params if m.startswith(("E2", "E3")) else embed_params if m.startswith("E1") else np.nan for m in results["model"]]
results["inference ms / 20 candidates"] = [COST.get(m, {}).get("inference ms / 20 candidates", np.nan) for m in results["model"]]
columns = ["model", "family", "nDCG@10", "MRR@10", "Recall@10", "ROC-AUC", "PR-AUC", "F1", "Precision", "Recall",
           "trainable params", "model size (params)", "train minutes", "inference ms / 20 candidates"]
results = results.reindex(columns=[c for c in columns if c in results.columns])
print(f"Test: {test_pairs['query_id'].nunique():,} synthetic questions, {len(test_pairs):,} pairs, {labels.mean():.1%} positive. "
      f"Ceiling (correct article among the {K_CAND} candidates): {ceiling:.1%}")
print(f"Inference time is on {DEVICE}; E1 rows exclude the embedding step ({embed_minutes:.1f} min for all texts).")
display(results.round(4))
results.to_csv(OUT / "results.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
results.set_index("model")[["nDCG@10", "MRR@10"]].plot.barh(ax=axes[0]); axes[0].invert_yaxis(); axes[0].axvline(ceiling, color="gray", ls="--", lw=1)
axes[0].set_title("Ranking on the test questions (dashed: ceiling)"); axes[0].set_ylabel("")
results.dropna(subset=["ROC-AUC"]).set_index("model")[["ROC-AUC", "PR-AUC", "F1"]].plot.barh(ax=axes[1]); axes[1].invert_yaxis()
axes[1].set_title("Pair classification on the test pairs"); axes[1].set_ylabel("")
plt.tight_layout(); plt.savefig(OUT / "results.png", dpi=150); plt.show()

### Statistical significance and breakdown

Paired tests on per-question nDCG@10 (the same test questions under both models): a 95% bootstrap confidence interval of the mean difference and a two-sided sign-flip randomisation test, 10,000 resamples each.

In [ ]:
def per_query_ndcg(name):
    frame = test_pairs.assign(score=SCORES[name]).sort_values(["query_id", "score"], ascending=[True, False])
    frame["rank"] = frame.groupby("query_id").cumcount() + 1
    ranks = frame[frame["label"] == 1].set_index("query_id")["rank"].reindex(test_pairs["query_id"].unique()).fillna(np.inf).astype(float)
    return pd.Series(np.where(ranks <= 10, 1 / np.log2(ranks + 1), 0.0), index=ranks.index), ranks

def paired_test(a, b, n=10_000):
    d = (per_query_ndcg(b)[0] - per_query_ndcg(a)[0]).to_numpy()
    rng = np.random.default_rng(SEED)
    boot = rng.choice(d, (n, len(d))).mean(1)
    flipped = (rng.choice([-1.0, 1.0], (n, len(d))) * d).mean(1)
    return {"comparison": f"{b}  vs  {a}", "mean nDCG@10 difference": d.mean(), "95% CI low": np.quantile(boot, 0.025),
            "95% CI high": np.quantile(boot, 0.975), "p-value": (np.abs(flipped) >= abs(d.mean()) - 1e-12).mean(),
            "questions better": int((d > 0).sum()), "worse": int((d < 0).sum()), "same": int((d == 0).sum())}

E0, E1, E2, E3 = "E0 TF-IDF features + LogReg", "E1c e5 cosine + lexical features + gradient boosting", "E2 cross-encoder, zero-shot", "E3 cross-encoder, fine-tuned"
E1_PURE = "E1b e5 embeddings + LogReg"
significance = pd.DataFrame([paired_test(E0, E1_PURE), paired_test(E0, E1), paired_test(E1, E2), paired_test(E2, E3), paired_test(E0, E3)])
display(significance.round(4)); significance.to_csv(OUT / "significance.csv", index=False)

meta = test_pairs.drop_duplicates("query_id").set_index("query_id")[["lang", "query_type"]]
by_group = pd.DataFrame({m: {f"{c}={v}": per_query_ndcg(m)[0].loc[ids].mean() for c in ("lang", "query_type") for v, ids in meta.groupby(c).groups.items()}
                         for m in (E0, E1_PURE, E1, E2, E3)}).T
display(by_group.round(3)); by_group.to_csv(OUT / "breakdown.csv")
by_group.T.plot.bar(figsize=(11, 4), rot=0); plt.ylabel("nDCG@10"); plt.title("Test nDCG@10 by language and question type")
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(OUT / "breakdown.png", dpi=150); plt.show()

## §8 Error analysis

Questions where the fine-tuned transformer (E3) does **not** put the correct article first, next to the rank the same article gets from E1. The automatic columns give the first hints about the cause; the error types are assigned by reading the rows (see the report).

In [ ]:
pd.set_option("display.max_colwidth", 110)
ndcg3, rank3 = per_query_ndcg(E3)
_, rank1 = per_query_ndcg(E1)
top = test_pairs.assign(score=SCORES[E3]).sort_values("score", ascending=False).drop_duplicates("query_id").set_index("query_id")
gold = queries.drop_duplicates("query_id").set_index("query_id")["article_id"]
titles = {lang: dict(zip(c["article_id"], c["title"])) for lang, c in corpus.items()}
first = test_pairs.drop_duplicates("query_id").set_index("query_id")
rows = []
for qid in rank3.index[rank3 > 1]:
    q = first.loc[qid]
    lang, gold_id = q["lang"], gold[qid]
    predicted_id = corpus[lang]["article_id"].iat[int(top.loc[qid, "doc"])]
    rows.append({"query": q["query"], "lang": lang, "type": q["query_type"],
                 "correct article": f"{gold_id}  {titles[lang].get(gold_id, '')}",
                 "E3 ranked first": f"{predicted_id}  {titles[lang].get(predicted_id, '')}",
                 "rank of correct: E3": rank3[qid], "E1": rank1[qid],
                 "not among candidates": bool(np.isinf(rank3[qid])),
                 "same code": gold_id.split(":")[0] == predicted_id.split(":")[0],
                 "question length (words)": len(q["query"].split())})
columns = ["query", "lang", "type", "correct article", "E3 ranked first", "rank of correct: E3", "E1",
           "not among candidates", "same code", "question length (words)"]
errors = pd.DataFrame(rows, columns=columns).sort_values(["not among candidates", "rank of correct: E3"], ascending=[True, False])
errors.to_csv(OUT / "errors.csv", index=False)
n = len(rank3)
print(f"E3 is wrong at rank 1 for {len(errors)} of {n} test questions ({len(errors) / n:.1%}):")
print(f"  - correct article not among the {K_CAND} BM25 candidates (the reranker cannot fix these): {int(errors['not among candidates'].sum())}")
print(f"  - correct article ranked 2-3: {int(errors['rank of correct: E3'].between(2, 3).sum())}; ranked 4-20: {int(errors['rank of correct: E3'].between(4, 20).sum())}")
print(f"  - wrong article from the same code: {int(errors['same code'].sum())}; from another code: {int((~errors['same code'].astype(bool)).sum())}")
print(f"  - by language: {errors['lang'].value_counts().to_dict()}; by question type: {errors['type'].value_counts().to_dict()}")
print(f"  - E1 had the correct article first in {int((errors['E1'] == 1).sum())} of these (E3 made it worse)")
print(f"Questions E1 got wrong at rank 1 and E3 got right: {int(((rank1 > 1) & (rank3 == 1)).sum())}")
display(errors.head(25))

## §9 Answers to the research questions

*Filled in from the tables above after the GPU run; every answer quotes the measured numbers.*

- **RQ1 (E1 vs E0).**
- **RQ2 (E2 vs E1).**
- **RQ3 (E3 vs E2).**
- **RQ4 (cost and the choice for the product).**
- **Limits.** LLM-written questions with one labelled article each; the reranker cannot exceed the BM25 ceiling; one seed, with bootstrap intervals in place of repeated runs.

In [ ]:
config = {"seed": SEED, "smoke": SMOKE, "device": DEVICE,
          "versions": {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__,
                       "scikit-learn": sklearn.__version__, "bm25s": bm25s.__version__},
          "data": {"train_pairs": len(train_pairs), "val_pairs": len(val_pairs), "test_pairs": len(test_pairs),
                   "test_questions": int(test_pairs["query_id"].nunique()), "candidates": K_CAND, "hard_negatives": N_NEG, "ceiling": float(ceiling)},
          "bm25": {lang: {"k1": k1, "b": b} for lang, (k1, b) in best_bm25.items()},
          "E0": {"model": "LogisticRegression(C=1, class_weight=balanced)", "features": FEATURES},
          "E1": {"embedding": EMB_NAME, "dimension": int(Q.shape[1]), "max_len": EMB_MAX_LEN, "logreg": lr_emb.best_params_, "gradient_boosting": gb.best_params_},
          "E2": {"model": CE_NAME, "max_len": MAX_LEN},
          "E3": {**FT, "history": history, "train_minutes": ft_minutes},
          "cost": COST}
(OUT / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

import shutil
archive = shutil.make_archive("a4_results", "zip", OUT)
print("Saved:", archive, f"({os.path.getsize(archive) / 1e6:.0f} MB): results, significance, breakdown, errors, config, charts and the fine-tuned reranker.")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass